# Serving an ML Model with FastAPI

FastAPI is a modern async web framework that auto-generates OpenAPI documentation and validates requests using Pydantic. This notebook builds a FastAPI model server and tests it in-notebook using `TestClient` — no uvicorn process required.

## Learning Objectives

By the end of this notebook you will be able to:
1. Explain FastAPI's key advantages over Flask (auto-docs, Pydantic, async support)
2. Define Pydantic models for request and response validation
3. Build async FastAPI endpoints for `/predict`, `/health`, and `/model-info`
4. Test a FastAPI app in-notebook using `TestClient`
5. Decide when to choose FastAPI vs Flask

## 1. Install Dependencies

In [ ]:
import subprocess, sys
subprocess.run([sys.executable, "-m", "pip", "install", "httpx", "-q"], check=True)
print("httpx ready (required by TestClient)")

## 2. Train and Save the Model

Same Iris RandomForest from notebook 01 — this lets us compare the serving layer directly without changing the model artifact.

In [ ]:
import joblib
import numpy as np
from sklearn.datasets import load_iris
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

iris = load_iris()
X_train, X_test, y_train, y_test = train_test_split(
    iris.data, iris.target, test_size=0.2, random_state=42
)

clf = RandomForestClassifier(n_estimators=100, random_state=42)
clf.fit(X_train, y_train)

acc = accuracy_score(y_test, clf.predict(X_test))
print(f"Test accuracy: {acc:.2%}")

MODEL_PATH = "/tmp/iris_rf_fastapi.joblib"
joblib.dump(clf, MODEL_PATH)
print(f"Model saved: {MODEL_PATH}")

## 3. Pydantic Models — Automatic Validation

Pydantic models describe the **shape** of request and response data. FastAPI uses them to:
- Automatically validate incoming JSON (wrong type or missing field → 422 Unprocessable Entity)
- Generate the OpenAPI schema that powers `/docs`
- Serialize response objects to JSON

You write Python type hints; Pydantic and FastAPI do the rest.

In [ ]:
from pydantic import BaseModel, Field
from typing import Dict

class IrisFeatures(BaseModel):
    sepal_length: float = Field(..., ge=0, description="Sepal length in cm")
    sepal_width:  float = Field(..., ge=0, description="Sepal width in cm")
    petal_length: float = Field(..., ge=0, description="Petal length in cm")
    petal_width:  float = Field(..., ge=0, description="Petal width in cm")

class PredictionResponse(BaseModel):
    prediction: str
    class_id: int
    confidence: float
    probabilities: Dict[str, float]

# Demonstrate parsing — Pydantic coerces and validates automatically
sample = IrisFeatures(sepal_length=5.1, sepal_width=3.5, petal_length=1.4, petal_width=0.2)
print("Valid input parsed:", sample)

# Uncomment to see validation error:
# bad = IrisFeatures(sepal_length="big", sepal_width=3.5, petal_length=1.4, petal_width=0.2)

## 4. Write the FastAPI Application

Notice `async def` on each endpoint. FastAPI runs on an async event loop (ASGI). Async endpoints shine when the handler `await`s on I/O — a database, a feature store, another HTTP service — because one worker can serve other requests while waiting.

**Important catch for ML serving:** a *blocking* call like sklearn's `predict()` inside an `async def` blocks the whole event loop, so it does **not** buy you concurrency and can even hurt it. FastAPI sidesteps this automatically if you declare the endpoint as a plain `def` — it runs that in a threadpool. So for CPU-bound inference, prefer a plain `def`, or offload the heavy call (e.g. `run_in_executor`). We use `async def` here to show the syntax; just don't assume `async` alone makes blocking work concurrent.

In [ ]:
%%writefile /tmp/fastapi_app.py
import joblib
import numpy as np
from fastapi import FastAPI, HTTPException
from pydantic import BaseModel, Field
from typing import Dict

app = FastAPI(
    title="Iris Classifier API",
    description="Classify Iris flowers using a RandomForest model",
    version="1.0.0"
)

# --- Pydantic schemas ---
class IrisFeatures(BaseModel):
    sepal_length: float = Field(..., ge=0, description="Sepal length in cm")
    sepal_width:  float = Field(..., ge=0, description="Sepal width in cm")
    petal_length: float = Field(..., ge=0, description="Petal length in cm")
    petal_width:  float = Field(..., ge=0, description="Petal width in cm")

class PredictionResponse(BaseModel):
    prediction: str
    class_id: int
    confidence: float
    probabilities: Dict[str, float]

# --- Load model once at startup ---
MODEL_PATH = "/tmp/iris_rf_fastapi.joblib"
model = joblib.load(MODEL_PATH)
CLASSES = ["setosa", "versicolor", "virginica"]

# --- Endpoints ---
@app.get("/health")
async def health():
    return {"status": "ok", "model_loaded": model is not None}

@app.get("/model-info")
async def model_info():
    return {
        "model_type": type(model).__name__,
        "n_estimators": model.n_estimators,
        "classes": CLASSES,
        "n_features": model.n_features_in_
    }

@app.post("/predict", response_model=PredictionResponse)
async def predict(features: IrisFeatures):
    try:
        X = np.array([[features.sepal_length, features.sepal_width,
                       features.petal_length, features.petal_width]])
        class_id  = int(model.predict(X)[0])
        probs     = model.predict_proba(X)[0]
        prob_dict = {cls: round(float(p), 4) for cls, p in zip(CLASSES, probs)}
        return PredictionResponse(
            prediction=CLASSES[class_id],
            class_id=class_id,
            confidence=round(float(probs.max()), 3),
            probabilities=prob_dict
        )
    except Exception as e:
        raise HTTPException(status_code=500, detail=str(e))

# Run with: uvicorn fastapi_app:app --host 0.0.0.0 --port 8000
# Docs at:  http://localhost:8000/docs

## 5. Test with TestClient

`TestClient` wraps the ASGI app and lets you make HTTP calls synchronously in a notebook — no uvicorn server needed. It uses `httpx` internally.

In [ ]:
import sys, importlib
sys.path.insert(0, "/tmp")
import fastapi_app as fmod
importlib.reload(fmod)

from fastapi.testclient import TestClient
client = TestClient(fmod.app)

# --- Health check ---
resp = client.get("/health")
print("Health:", resp.status_code, resp.json())

# --- Model info ---
resp = client.get("/model-info")
print("Model info:", resp.status_code, resp.json())

# --- Valid prediction ---
payload = {"sepal_length": 5.1, "sepal_width": 3.5, "petal_length": 1.4, "petal_width": 0.2}
resp = client.post("/predict", json=payload)
print("Predict (setosa):", resp.status_code, resp.json())

# --- Missing field — Pydantic raises 422 automatically ---
resp = client.post("/predict", json={"sepal_length": 5.1})
print("Missing fields -> 422:", resp.status_code)

# --- Wrong type — Pydantic catches it ---
bad = {"sepal_length": "big", "sepal_width": 3.5, "petal_length": 1.4, "petal_width": 0.2}
resp = client.post("/predict", json=bad)
print("Wrong type -> 422:", resp.status_code)

## 6. The Auto-Generated `/docs` Endpoint

When you run `uvicorn fastapi_app:app`, visiting `http://localhost:8000/docs` shows an interactive Swagger UI. You can try requests directly in the browser, see expected inputs/outputs, and download the OpenAPI spec from `/openapi.json`. This is generated automatically from your Pydantic models — no extra work.

In [ ]:
import json

# Inspect the auto-generated OpenAPI schema
resp = client.get("/openapi.json")
schema = resp.json()

print("API title:", schema["info"]["title"])
print("Version:", schema["info"]["version"])
print("Endpoints:", list(schema["paths"].keys()))

print("\nIrisFeatures schema (auto-generated from Pydantic):")
iris_schema = schema["components"]["schemas"]["IrisFeatures"]
print(json.dumps({"required": iris_schema.get("required", []),
                  "properties": list(iris_schema["properties"].keys())}, indent=2))

## 7. Flask vs FastAPI Decision Table

| Scenario | Choose |
|---|---|
| Team already uses Flask, simple API | Flask |
| Need auto-generated docs for a frontend team | FastAPI |
| High concurrent I/O-bound traffic | FastAPI (async) |
| Integrating into an existing Django codebase | Flask |
| New greenfield ML microservice | FastAPI |
| Strict request/response type safety | FastAPI (Pydantic) |

## 8. Summary

In this notebook you:
- Defined `IrisFeatures` and `PredictionResponse` Pydantic models for automatic validation
- Built three async FastAPI endpoints: `/health`, `/model-info`, `/predict`
- Tested the app in-notebook using `TestClient` — no uvicorn process needed
- Inspected the auto-generated OpenAPI schema from `/openapi.json`

FastAPI's key insight: **write Python type hints once, get validation + serialization + documentation for free.**

## Self-Check (answer before scrolling up)

1. **What does Pydantic do when a required field is missing from the request body?** What HTTP status code does FastAPI return automatically?
2. **What is the `/docs` endpoint and how does it help during development?** Name one thing you can do there that you can't do with just curl.
3. **When would you choose Flask instead of FastAPI?** Give a concrete real-world scenario.